# Подготовка данных

In [4]:
import os
import random
import numpy as np
import torch
import kagglehub
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed = 42
seed_everything(seed)

In [ ]:
# Скачивание датасета
path = kagglehub.dataset_download("zachaluza/cnfood-241")
print("Путь к датасету:", path)

In [6]:
def create_food_dataloaders(dataset_path, batch_size=32, num_workers=2, subset_ratio=1.0, seed=42):
    train_dir = os.path.join(dataset_path, 'train600x600')
    val_dir = os.path.join(dataset_path, 'val600x600')

    train_transforms = T.Compose([
        T.Resize(256),
        T.RandomResizedCrop(224),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
    full_val_dataset = ImageFolder(root=val_dir, transform=val_transforms)
    classes = full_train_dataset.classes

    train_dataset = full_train_dataset
    val_dataset = full_val_dataset

    if subset_ratio < 1.0:
        train_idx = list(range(len(full_train_dataset)))
        _, train_indices = train_test_split(
            train_idx, test_size=subset_ratio,
            stratify=full_train_dataset.targets, random_state=seed
        )
        val_idx = list(range(len(full_val_dataset)))
        _, val_indices = train_test_split(
            val_idx, test_size=subset_ratio,
            stratify=full_val_dataset.targets, random_state=seed
        )

        train_dataset = Subset(full_train_dataset, train_indices)
        val_dataset = Subset(full_val_dataset, val_indices)

    print(f"Обучение (Train): {len(train_dataset)} изобр.")
    print(f"Валидация (Val): {len(val_dataset)} изобр.")
    print(f"Количество классов: {len(classes)}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True, persistent_workers=True
    )

    return train_loader, val_loader, classes


In [ ]:
train_loader, val_loader, class_names = create_food_dataloaders(path,batch_size=256, num_workers = 20, subset_ratio=1.0, seed=seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

# Оценка

In [8]:
from tqdm.notebook import tqdm

def evaluate_model_topk(model, dataloader, device, is_psd=False):
    model.eval()
    correct_top1 = 0
    correct_top5 = 0
    total_samples = 0

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating Top-K", leave=False):
            images, targets = images.to(device), targets.to(device)

            if is_psd:
                logits = PSDModel.get_teacher_logits(model, images)
            else:
                logits = model(images)

            maxk = min(5, logits.size(-1))
            _, pred = logits.topk(maxk, dim=1, largest=True, sorted=True)
            pred = pred.t()
            correct = pred.eq(targets.view(1, -1).expand_as(pred))

            correct_top1 += correct[0].view(-1).float().sum().item()
            correct_top5 += correct[:5].reshape(-1).float().sum().item()
            total_samples += targets.size(0)

    top1_acc = (correct_top1 / total_samples) * 100
    top5_acc = (correct_top5 / total_samples) * 100
    return top1_acc, top5_acc

In [9]:
import pandas as pd

def log_epoch_metrics(model_name, epoch, train_loss, train_acc, val_loss, val_acc):
    log_dir = "training_logs"
    os.makedirs(log_dir, exist_ok=True)
    filename = os.path.join(log_dir, f"{model_name}_metrics.csv")

    new_data = pd.DataFrame([{
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    }])

    if os.path.exists(filename):
        new_data.to_csv(filename, mode='a', header=False, index=False)
    else:
        new_data.to_csv(filename, mode='w', header=True, index=False)

# Базовая модель ConvNeXt

In [ ]:
from torch.cuda.amp import GradScaler
USE_BFLOAT16 = False

if USE_BFLOAT16:
    amp_dtype = torch.bfloat16
    scaler = None
else:
    amp_dtype = torch.float16
    scaler = GradScaler()

In [12]:
import torch.nn as nn
import torchvision.models as models
from torch.cuda.amp import autocast

def get_baseline_convnext(model_name="convnext_tiny"):
    if model_name == "convnext_tiny":
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    elif model_name == "convnext_base":
        model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.DEFAULT)
    else:
        raise ValueError("Неизвестная модель")

    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, 241)
    return model

def train_baseline_epoch(model, dataloader, optimizer, criterion, device, epoch, amp_dtype, scaler):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(dataloader, desc=f"Baseline Training {epoch+1} эпоха")
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast(dtype=amp_dtype):
            logits = model(images)
            loss = criterion(logits, targets)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(logits, 1)
        correct += (predicted == targets).sum().item()
        total += targets.size(0)
        pbar.set_postfix({'Loss': f"{loss.item():.4f}", 'Acc': f"{(correct/total)*100:.2f}%"})

    return running_loss / total, (correct / total) * 100

def validate_baseline(model, dataloader, criterion, device, amp_dtype):
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, targets in dataloader:
            images, targets = images.to(device), targets.to(device)

            with autocast(dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, targets)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(logits, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    return val_loss / total, (correct / total) * 100

In [ ]:
NUM_EPOCHS = 15
baseline_model = get_baseline_convnext('convnext_tiny').to(device)
optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=3e-4, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

scaler = GradScaler()
print("\n Обучение базовой модели ")
epoch_pbar = tqdm(range(NUM_EPOCHS), desc="Общий прогресс обучения")
for epoch in epoch_pbar:
    t_loss, t_acc = train_baseline_epoch(baseline_model, train_loader, optimizer, criterion, device, epoch, amp_dtype, scaler)
    v_loss, v_acc = validate_baseline(baseline_model, val_loader, criterion, device, amp_dtype)
    log_epoch_metrics("Base ConvNeXt", epoch, t_loss, t_acc, v_loss, v_acc)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Эпоха {epoch+1} -> Train Acc: {t_acc:.2f}% | Val Acc: {v_acc:.2f}%")

base_top1, base_top5 = evaluate_model_topk(baseline_model, val_loader, device)
print(f"\n[ИТОГ] Val Top-1: {base_top1:.2f}% | Val Top-5: {base_top5:.2f}%")

In [ ]:
torch.save(baseline_model.state_dict(), "baseline_convnext_tiny.pth")

# PSD модель

In [14]:
import torch.nn.functional as F

class PSDModel(nn.Module):
    def __init__(self, model_name='convnext_tiny', num_classes=241, num_students=2):
        super().__init__()
        self.num_students = num_students
        self.num_classes = num_classes

        if model_name == 'convnext_tiny':
            self.backbone = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
            in_features_teacher = 768
            configs = {
                2: ([3, 5], [192, 384]),
                3: ([1, 3, 5], [96, 192, 384])
            }
        else:
            raise ValueError("Допустимая модель: 'convnext_tiny'")

        self.feature_cutoffs, in_features_list = configs[num_students]

        self.crm_classifiers = nn.ModuleList([
            nn.Linear(in_dim, num_classes) for in_dim in in_features_list
        ])

        self.heads_student = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm((in_features_teacher, 1, 1), eps=1e-6),
                nn.Flatten(1),
                nn.Linear(in_features_teacher, num_classes)
            ) for _ in range(num_students)
        ])

        if isinstance(self.backbone.classifier[2], nn.Linear):
            self.backbone.classifier[2] = nn.Linear(in_features_teacher, num_classes)
        self.head_teacher = self.backbone.classifier

    def forward_features_to_stage(self, x, stage_idx):
        # Прогон тензора от начала до указанного индекса слоя features
        for i, layer in enumerate(self.backbone.features):
            x = layer(x)
            if i == stage_idx:
                break
        return x

    def forward_features_from_stage_to_end(self, x, start_stage_idx):
        # Прогон тензора до конца экстрактора
        start_trigger = False
        for i, layer in enumerate(self.backbone.features):
            if start_trigger:
                x = layer(x)
            if i == start_stage_idx:
                start_trigger = True

        if x.dim() == 4 and x.shape[-1] > x.shape[1]:
            x = x.permute(0, 3, 1, 2)

        return x

    def generate_crm_mask(self, features, labels, stage_list_idx, quantile_val=0.7):
        b = features.shape[0]

        if features.dim() == 4 and features.shape[-1] > features.shape[1]:
            h, w, c = features.shape[1], features.shape[2], features.shape[3]
            features_cf = features.permute(0, 3, 1, 2)
        else:
            c, h, w = features.shape[1], features.shape[2], features.shape[3]
            features_cf = features

        weight = self.crm_classifiers[stage_list_idx].weight.detach()
        assert weight.shape[1] == c, f"Ошибка размерности каналов CRM"

        target_weights = weight[labels].view(b, c, 1, 1)
        cam = torch.sum(features_cf * target_weights, dim=1, keepdim=True)
        cam = F.relu(cam)

        cam_flat = cam.view(b, -1)
        thresholds = torch.quantile(cam_flat, quantile_val, dim=1, keepdim=True)

        # маскирование
        mask = (cam.view(b, -1) < thresholds).float().view(b, 1, h, w)
        return mask

    @staticmethod
    def get_teacher_logits(self, x):
        features = self.backbone.features(x)
        pooled = self.backbone.avgpool(features)
        return self.head_teacher(pooled)

    @staticmethod
    def get_teacher_confidence(logits_teacher, num_classes=241):
        prob_teacher = F.softmax(logits_teacher, dim=-1)
        entropy = -torch.sum(prob_teacher * torch.log(prob_teacher + 1e-9), dim=-1)
        max_entropy = torch.log(torch.tensor(float(num_classes), device=logits_teacher.device))
        norm_entropy = entropy / max_entropy
        return (1.0 - norm_entropy).unsqueeze(-1)

In [15]:
import math

def compute_psd_loss(student_logits, logits_teacher, targets, criterion_ce, beta=1.0):
    loss_ce_teacher = criterion_ce(logits_teacher, targets)
    loss_ce_students = sum([criterion_ce(lg, targets) for lg in student_logits])

    loss_kl = 0.0
    prob_teacher = F.softmax(logits_teacher, dim=-1)
    for lg in student_logits:
        log_p_s = F.log_softmax(lg, dim=-1)
        loss_kl += F.kl_div(log_p_s, prob_teacher, reduction='batchmean')

    return loss_ce_teacher + loss_ce_students + (beta * loss_kl)

def train_epoch_psd(model, train_loader, optimizer, current_epoch, device, criterion_ce, scaler, amp_dtype, accumulation_steps=1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    temperature = 3.0
    alpha_param = 1.0
    beta_param = 5
    omega_l = 1.0

    # Warm-up дистилляции
    if current_epoch < beta_param:
        omega_d = alpha_param * math.exp(-5 * (1 - current_epoch / beta_param) ** 2)
    else:
        omega_d = alpha_param

    criterion_kl = nn.KLDivLoss(reduction='batchmean')
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"PSD | Эпоха {current_epoch+1}")

    for batch_idx, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)

        with autocast(dtype=amp_dtype):
            # Шаг учителя
            first_cutoff = model.feature_cutoffs[0]
            features_stage_original = model.forward_features_to_stage(images, first_cutoff)
            final_features_original = model.forward_features_from_stage_to_end(features_stage_original, first_cutoff)

            pooled_teacher = model.backbone.avgpool(final_features_original)
            outputs_teacher = model.head_teacher(pooled_teacher)

            loss_g = criterion_ce(outputs_teacher, labels)
            p_teacher = F.softmax(outputs_teacher / temperature, dim=1).detach()

            sum_loss_l = 0.0
            sum_loss_d = 0.0
            current_images = images.clone()

            # Каскад по студентам
            for i in range(model.num_students):
                cutoff_idx = model.feature_cutoffs[i]
                current_features_stage = model.forward_features_to_stage(current_images, cutoff_idx)

                if current_features_stage.dim() == 4 and current_features_stage.shape[-1] > current_features_stage.shape[1]:
                    current_features_stage = current_features_stage.permute(0, 3, 1, 2)

                pooled_crm = F.adaptive_avg_pool2d(current_features_stage, (1, 1)).flatten(1)
                outputs_crm = model.crm_classifiers[i](pooled_crm)
                sum_loss_l += criterion_ce(outputs_crm, labels)

                # Маскирование
                crm_mask_low = model.generate_crm_mask(current_features_stage, labels, stage_list_idx=i)
                crm_mask_high = F.interpolate(crm_mask_low, size=(images.shape[2], images.shape[3]), mode='bilinear', align_corners=False)

                current_images = current_images * crm_mask_high

                # Проход студента
                features_s_masked = model.forward_features_to_stage(current_images, cutoff_idx)
                final_features_student = model.forward_features_from_stage_to_end(features_s_masked, cutoff_idx)

                pooled_student = model.backbone.avgpool(final_features_student)
                outputs_student = model.heads_student[i](pooled_student)

                log_p_student = F.log_softmax(outputs_student / temperature, dim=1)
                sum_loss_d += criterion_kl(log_p_student, p_teacher) * (temperature ** 2)

            total_loss = (loss_g + (omega_l * sum_loss_l / model.num_students) + (omega_d * sum_loss_d / model.num_students)) / accumulation_steps

        if scaler is not None:
            scaler.scale(total_loss).backward()
            if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            total_loss.backward()
            if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad()

        running_loss += total_loss.item() * accumulation_steps
        _, predicted = outputs_teacher.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({
            'Loss': f"{total_loss.item() * accumulation_steps:.4f}",
            'Acc': f"{100. * correct / total:.2f}%"
        })

    return running_loss / len(train_loader), 100. * correct / total

def validate_psd(model, dataloader, criterion_ce, device, amp_dtype):
    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Validating", leave=False):
            images, targets = images.to(device), targets.to(device)

            logits_teacher = model.get_teacher_logits(images)
            loss = criterion_ce(logits_teacher, targets)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(logits_teacher, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)

    return val_loss / total, (correct / total) * 100

In [ ]:
NUM_EPOCHS = 15

print("\n Обучение PSD ")
model_original = PSDModel(model_name='convnext_tiny', num_classes=241, num_students=2).to(device)

criterion_ce = nn.CrossEntropyLoss()
scaler = GradScaler()

optimizer = torch.optim.AdamW(
    model_original.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

epoch_pbar = tqdm(range(NUM_EPOCHS), desc="Общий прогресс обучения")
for epoch in epoch_pbar:

    train_loss, train_acc = train_epoch_psd(
        model_original, train_loader, optimizer, epoch, device, criterion_ce, scaler, amp_dtype
    )

    val_loss, val_acc = validate_psd(model_original, val_loader, criterion_ce, device, amp_dtype)
    log_epoch_metrics("ConvNeXt_PSD", epoch, train_loss, train_acc, val_loss, val_acc)
    scheduler.step()

    current_lr = scheduler.get_last_lr()[0]
    print(f"Эпоха {epoch+1:02d}/{NUM_EPOCHS:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {current_lr}")

orig_top1, orig_top5 = evaluate_model_topk(model_original, val_loader, device, is_psd=True)
print(f"\n[ИТОГ PSD] Val Top-1: {orig_top1:.2f}% | Val Top-5: {orig_top5:.2f}%")

In [ ]:
orig_top1, orig_top5 = evaluate_model_topk(model_original, val_loader, device, is_psd=True)
print(f"\n[ИТОГ PSD] Val Top-1: {orig_top1:.2f}% | Val Top-5: {orig_top5:.2f}%")

In [ ]:
torch.save(model_original.state_dict(), "convnext_psd.pth")

# Графики

In [30]:
import matplotlib.pyplot as plt

def plot_models_comparison(model_names=None):
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

    plt.rcParams.update({
        'font.size': 16,
        'axes.labelsize': 18,
        'xtick.labelsize': 15,
        'ytick.labelsize': 15,
        'legend.fontsize': 14,
        'lines.markersize': 8,
        'lines.linewidth': 3,
        'figure.titlesize': 20
    })

    if isinstance(model_names, str):
        model_names = [model_names]

    if model_names is None:
        model_names = ["ConvNeXt_PSD", "Base ConvNeXt"]

    log_dir = "training_logs"
    os.makedirs(log_dir, exist_ok=True)

    colors = {'ConvNeXt_PSD': '#1f77b4', 'Base ConvNeXt': '#d62728'}

    data = {}
    for name in model_names:
        filepath = os.path.join(log_dir, f"{name}_metrics.csv")
        if os.path.exists(filepath):
            data[name] = pd.read_csv(filepath)
        else:
            print(f"Файл истории для {name} не найден")

    if not data:
        print("Нет данных для отображения графиков")
        return

    # Loss
    fig1, ax1 = plt.subplots(figsize=(7, 5.5))

    for name, df in data.items():
        epochs = df['epoch']
        color = colors.get(name, None)
        ax1.plot(epochs, df['train_loss'], linestyle='--', marker='o', alpha=0.5, color=color, label=f'{name} (Train)')
        ax1.plot(epochs, df['val_loss'], linestyle='-', marker='o', color=color, label=f'{name} (Val)') # Разные маркеры для Val

    ax1.set_xlabel('Эпоха')
    ax1.set_ylabel('Значение потери')
    ax1.xaxis.get_major_locator().set_params(integer=True)
    ax1.legend(loc='upper right', frameon=True)
    ax1.grid(True, alpha=0.4, linestyle=':')

    plt.tight_layout()
    loss_path = os.path.join(log_dir, 'models_comparison_plot_loss-1.png')
    plt.savefig(loss_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig1)

    # Accuracy
    fig2, ax2 = plt.subplots(figsize=(7, 5.5))

    for name, df in data.items():
        epochs = df['epoch']
        color = colors.get(name, None)
        ax2.plot(epochs, df['train_acc'], linestyle='--', marker='o', alpha=0.5, color=color, label=f'{name} (Train)')
        ax2.plot(epochs, df['val_acc'], linestyle='-', marker='o', color=color, label=f'{name} (Val)')

    ax2.set_xlabel('Эпоха')
    ax2.set_ylabel('Точность (%)')
    ax2.xaxis.get_major_locator().set_params(integer=True)
    ax2.legend(loc='lower right', frameon=True)
    ax2.grid(True, alpha=0.4, linestyle=':')

    plt.tight_layout()
    acc_path = os.path.join(log_dir, 'models_comparison_plot_acc-1.png')
    plt.savefig(acc_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig2)


In [ ]:
plot_models_comparison()

In [32]:
def print_model_summary(model):
    print("\n" + "="*40)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Всего параметров (Total params):  {total_params:,}")
    print(f"Обучаемых параметров (Trainable): {trainable_params:,}")
    print("="*40 + "\n")

In [ ]:
def load_saved_model(checkpoint_path, model_name="convnext_tiny", num_classes=241, device="cuda"):
    if model_name == "convnext_tiny":
        model = models.convnext_tiny(weights=None)
    elif model_name == "convnext_base":
        model = models.convnext_base(weights=None)
    else:
        raise ValueError("Неизвестная модель")

    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)

    raw_checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(raw_checkpoint, dict) and 'model_state_dict' in raw_checkpoint:
        state_dict = raw_checkpoint['model_state_dict']
    else:
        state_dict = raw_checkpoint

    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith("backbone."):
            new_key = k[9:]
            clean_state_dict[new_key] = v
        elif not any(x in k for x in ["crm_classifiers", "heads_student", "head_teacher"]):
            clean_state_dict[k] = v

    model.load_state_dict(clean_state_dict, strict=True)
    print(f"Успешно извлечен бэкбон из PSD модели и загружен в стандартный {model_name}!")

    model = model.to(device)
    model.eval()
    return model

In [33]:
from sklearn.metrics import f1_score, classification_report

def evaluate_model_f1(model, dataloader, device):
    model.eval()

    all_targets = []
    all_predictions = []

    print("Запуск сбора предсказаний на валидационной выборке...")
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Инференс"):
            images, targets = images.to(device), targets.to(device)

            with autocast(dtype=torch.bfloat16):
                logits = model(images)

            _, predicted_classes = torch.max(logits, dim=1)

            all_targets.extend(targets.cpu().numpy())
            all_predictions.extend(predicted_classes.cpu().numpy())

    all_targets = np.array(all_targets)
    all_predictions = np.array(all_predictions)

    macro_f1 = f1_score(all_targets, all_predictions, average='macro') * 100

    print(f"\n Macro F1-SCORE: {macro_f1:.2f}%")

    report = classification_report(all_targets, all_predictions, digits=4)
    print("\n".join(report.split("\n")[:15]))

    return macro_f1

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_saved_model("convnext_psd.pth", model_name="convnext_tiny", device=device)
print_model_summary(model)
metrics = evaluate_model_f1(model, val_loader, device)